# MEDDIAG Training on Kaggle (Free GPU)

**Purpose:** Resume Stage 2 LoRA training from checkpoint (step 2000+)

**Prerequisites:**
- Kaggle Secrets: `HF_TOKEN` and `SEMANTIC_SCHOLAR_API_KEY` must be enabled for this notebook
- Kaggle Dataset `meddiag-weights` added as input (contains checkpoint + FAISS index)
- Accelerator set to **GPU P100** or **T4** in notebook settings
- Internet **On** in notebook settings

**Expected runtime per session:** 6–10 hours (P100 16 GB)  
**Steps remaining from step 2000:** ~5 500 steps (~2–3 Kaggle sessions)

See `KAGGLE_SETUP.md` in the repo for full setup instructions.

---

## Step 1: Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'WARNING: No GPU detected. Enable GPU in notebook settings.')

## Step 2: Install Dependencies

Kaggle already has PyTorch with CUDA — we only install the extra packages.  
Pinned versions match the RunPod setup for reproducibility.

In [ ]:
!pip install -q \
    transformers==4.57.6 \
    peft==0.19.1 \
    bitsandbytes==0.49.2 \
    datasets \
    accelerate \
    sentence-transformers \
    faiss-cpu \
    scikit-learn \
    pillow \
    requests \
    python-dotenv \
    bert-score \
    gdown

print('Dependencies installed.')

## Step 3: Load Kaggle Secrets

Secrets must be toggled ON for this notebook in **Add-ons → Secrets**.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['SEMANTIC_SCHOLAR_API_KEY'] = secrets.get_secret('SEMANTIC_SCHOLAR_API_KEY')
    print('Secrets loaded from Kaggle.')
except Exception as e:
    print(f'WARNING: Could not load Kaggle secrets ({e})')
    print('If running outside Kaggle, set HF_TOKEN manually below:')
    # os.environ['HF_TOKEN'] = 'your_token_here'

# Set VRAM limit to 14 GB (leaves headroom on P100/T4 16 GB)
os.environ['MEDDIAG_MAX_VRAM_GB'] = '14'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

token_preview = os.environ.get('HF_TOKEN', '')[:10]
print(f'HF_TOKEN prefix: {token_preview}...' if token_preview else 'ERROR: HF_TOKEN not set — training will fail.')

## Step 4: Clone Repository

In [ ]:
import os

REPO_DIR = '/kaggle/working/visual-language-model-research-qlora-cot-rag'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git /kaggle/working/visual-language-model-research-qlora-cot-rag
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## Step 5: Load Model Weights

This cell checks three sources in priority order:
1. **Kaggle Dataset input** (`/kaggle/input/meddiag-weights/`) — fastest, recommended
2. **Previous session output** (`/kaggle/input/meddiag-ckpt-*/`) — for resuming across sessions
3. **Google Drive** via gdown — fallback if no Kaggle Dataset is attached

**To use the Kaggle Dataset:** Add `meddiag-weights` as input in the notebook sidebar.  
**To use Google Drive:** Set `GDRIVE_FOLDER_ID` below (get ID from the share link).

In [ ]:
import os
import shutil
import glob

# ─── Configuration ────────────────────────────────────────────────────────────
GDRIVE_FOLDER_ID = ''   # Set this if using Google Drive fallback (ask Sujal for the ID)
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('models', exist_ok=True)
os.makedirs('faiss_index', exist_ok=True)

def _copy_if_missing(src, dst):
    if os.path.exists(dst):
        print(f'  Already exists: {dst}')
        return
    if os.path.isdir(src):
        shutil.copytree(src, dst)
    else:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  Copied: {src} -> {dst}')

# ── Source 1: Kaggle Dataset input (meddiag-weights) ─────────────────────────
kaggle_weights = '/kaggle/input/meddiag-weights'
if os.path.exists(kaggle_weights):
    print('Loading weights from Kaggle Dataset input...')
    for fname in ['projector_stage1.pt', 'cls_head.pt']:
        src = os.path.join(kaggle_weights, fname)
        if os.path.exists(src):
            _copy_if_missing(src, os.path.join('models', fname))
    # Copy checkpoint folder (lora_step2000 or latest)
    for ckpt_dir in sorted(glob.glob(os.path.join(kaggle_weights, 'lora_step*'))):
        name = os.path.basename(ckpt_dir)
        _copy_if_missing(ckpt_dir, os.path.join('models', name))
    # Copy FAISS index
    faiss_src = os.path.join(kaggle_weights, 'faiss_index')
    if os.path.exists(faiss_src):
        _copy_if_missing(faiss_src, 'faiss_index')
    print('Weights loaded from Kaggle Dataset.')

# ── Source 2: Previous session checkpoint output ───────────────────────────────
prev_ckpts = sorted(glob.glob('/kaggle/input/meddiag-ckpt-*/models/lora_step*'))
if prev_ckpts:
    latest = max(prev_ckpts, key=lambda p: int(p.split('lora_step')[-1].split('/')[0]))
    ckpt_name = os.path.basename(latest)
    dst = os.path.join('models', ckpt_name)
    print(f'Found previous session checkpoint: {ckpt_name}')
    _copy_if_missing(latest, dst)

# ── Source 3: Google Drive fallback ───────────────────────────────────────────
if GDRIVE_FOLDER_ID and not os.path.exists('models/lora_step2000'):
    print('Downloading weights from Google Drive...')
    !gdown --folder "https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}" -O /tmp/gdrive_weights
    for fname in ['projector_stage1.pt', 'cls_head.pt']:
        src = f'/tmp/gdrive_weights/{fname}'
        if os.path.exists(src):
            _copy_if_missing(src, os.path.join('models', fname))
    for ckpt_dir in sorted(glob.glob('/tmp/gdrive_weights/lora_step*')):
        _copy_if_missing(ckpt_dir, os.path.join('models', os.path.basename(ckpt_dir)))
    faiss_src = '/tmp/gdrive_weights/faiss_index'
    if os.path.exists(faiss_src):
        _copy_if_missing(faiss_src, 'faiss_index')
    print('Weights downloaded from Google Drive.')

print('\nWeight loading complete.')

## Step 6: Detect Latest Checkpoint & Verify Files

In [ ]:
import glob
import os

# Auto-detect the latest available checkpoint
ckpt_dirs = sorted(
    glob.glob('models/lora_step*'),
    key=lambda p: int(p.split('lora_step')[-1])
)

if not ckpt_dirs:
    raise FileNotFoundError(
        'No lora_step* checkpoint found in models/. '
        'Check Step 5 — weights may not have loaded correctly.'
    )

RESUME_FROM = ckpt_dirs[-1]  # use highest step number
print(f'Will resume from: {RESUME_FROM}')

# Verify critical files
required = [
    'models/projector_stage1.pt',
    f'{RESUME_FROM}/adapter_model.safetensors',
    f'{RESUME_FROM}/train_state.pt',
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print('MISSING FILES:')
    for f in missing:
        print(f'  {f}')
    raise FileNotFoundError('Cannot train — required checkpoint files are missing.')

print('All required files verified.')

# Set pipeline state so the pipeline knows stages 0-2 are already done
os.makedirs('logs', exist_ok=True)
with open('logs/.pipeline_state', 'w') as f:
    f.write('step0\nstep1\nstep2\n')
print('Pipeline state set (stages 0, 1, 2 complete).')

# Summary
step = int(RESUME_FROM.split('lora_step')[-1])
remaining = max(0, 7500 - step)
print(f'\nResuming from step {step}. ~{remaining} steps remaining.')

## Step 7: Run Stage 2 Training

This cell **blocks until training completes or the session times out** (up to 12 hours).  
Output streams live to the cell. Checkpoints are saved every 250 steps automatically.

If the session times out mid-training:
1. The latest checkpoint is in `/kaggle/working/visual-language-model-research-qlora-cot-rag/models/lora_stepXXXX/`
2. Kaggle auto-saves `/kaggle/working/` as the notebook output on session end
3. Create a Dataset from the output and add it as input in the next session
4. Re-run all cells — Step 6 will auto-detect the new checkpoint

**Memory settings for P100 / T4 (16 GB VRAM):**
- `--grad-accum-steps 8` (effective batch size = 8)
- `--max-pairs 500` (balanced pairs per epoch, keeps peak VRAM ~13 GB)
- `MEDDIAG_MAX_VRAM_GB=14` (set in Step 3)

In [ ]:
import subprocess
import sys
import os

# RESUME_FROM is set by Step 6
cmd = [
    sys.executable, '-m', 'experiments.stage2_classification',
    '--projector-path',    'models/projector_stage1.pt',
    '--max-pairs',         '500',
    '--epochs',            '3',
    '--lr',                '2e-4',
    '--warmup-steps',      '50',
    '--grad-accum-steps',  '8',
    '--save-every',        '250',
    '--log-every',         '25',
    '--lora-save-dir',     'models/lora_adapter',
    '--resume-from',       RESUME_FROM,
]

print('Starting training with command:')
print(' '.join(cmd))
print('─' * 60)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
    cwd=REPO_DIR,
    env=os.environ,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print('─' * 60)
print(f'Training exited with code: {proc.returncode}')
if proc.returncode == 0:
    print('Training completed successfully.')
else:
    print('Training stopped early (OOM, timeout, or error). Latest checkpoint is saved.')

## Step 8: Monitor Training Progress

Run this cell **in a separate tab** while training runs in Step 7.  
Kaggle does not support true parallel cells — open a second notebook session
or check after training finishes.

In [ ]:
import json
import os

log_file = 'logs/stage2.jsonl'

if not os.path.exists(log_file):
    print('Training logs not found yet — has training started?')
else:
    logs = []
    with open(log_file) as f:
        for line in f:
            try:
                logs.append(json.loads(line))
            except json.JSONDecodeError:
                pass

    if not logs:
        print('Log file exists but is empty — training may not have started yet.')
    else:
        latest = logs[-1]
        hours = latest.get('elapsed_s', 0) / 3600
        step = latest.get('step', 0)
        total_steps = 7500
        pct = step / total_steps * 100

        print(f'=== Training Progress ===')
        print(f'Step:       {step} / {total_steps}  ({pct:.1f}%)')
        print(f'Epoch:      {latest.get("epoch", "?")}')
        print(f'Loss:       {latest.get("loss", "?")}')
        print(f'VRAM:       {latest.get("vram_gb", "?"):.2f} GB')
        print(f'Elapsed:    {hours:.1f} hours')

        if step > 0 and 'elapsed_s' in latest:
            secs_per_step = latest['elapsed_s'] / step
            remaining_steps = total_steps - step
            remaining_hrs = remaining_steps * secs_per_step / 3600
            print(f'Remaining:  ~{remaining_hrs:.1f} hours ({remaining_hrs/12:.1f} Kaggle sessions)')

        print(f'\n=== Last 5 Log Entries ===')
        for entry in logs[-5:]:
            print(f"step={entry.get('step')}  loss={entry.get('loss', '?'):.4f}  lr={entry.get('lr', '?'):.2e}")

## Step 9: Checkpoint Summary (Run After Training or Before Session Ends)

In [ ]:
import glob
import os

print('=== Saved Checkpoints ===')
ckpts = sorted(
    glob.glob('models/lora_step*'),
    key=lambda p: int(p.split('lora_step')[-1])
)
for ckpt in ckpts:
    files = os.listdir(ckpt)
    size_mb = sum(os.path.getsize(os.path.join(ckpt, f)) for f in files) / 1024**2
    print(f'  {ckpt:<35}  {size_mb:.1f} MB  files: {files}')

if ckpts:
    latest = ckpts[-1]
    step = int(latest.split('lora_step')[-1])
    print(f'\nLatest checkpoint: {latest} (step {step})')
    print(f'\nThis checkpoint is in /kaggle/working/ and will be auto-saved when the session ends.')
    print(f'To resume next session:')
    print(f'  1. Go to notebook Output tab → Create Dataset from Output')
    print(f'  2. Name it: meddiag-ckpt-step{step}')
    print(f'  3. Add that dataset as input in next session')
    print(f'  4. Run all cells — Step 6 will auto-detect step {step}')
else:
    print('No checkpoints found in models/.')

## Troubleshooting

### `HF_TOKEN not set` error
- Go to Add-ons → Secrets → make sure `HF_TOKEN` is toggled **On** for this notebook

### Out of memory (OOM)
- Kaggle P100/T4 has 16 GB VRAM; the model uses ~12–14 GB with 4-bit quant
- Try reducing `--max-pairs` to `200` in Step 7 if OOM persists
- Check VRAM with `!nvidia-smi` between cells

### `No lora_step* checkpoint found`
- The Kaggle Dataset `meddiag-weights` may not be attached. Go to notebook sidebar → Add Data
- Or set `GDRIVE_FOLDER_ID` in Step 5 to fall back to Google Drive download

### Session timed out before training finished
- This is expected. Follow the resume workflow in Step 9
- Training auto-saves every 250 steps — you lose at most 250 steps of progress

### bitsandbytes CUDA error
- Run `!pip install -q bitsandbytes==0.49.2 --force-reinstall` and restart the kernel

### Training is very slow
- P100 is slower than RTX 3090. Each step takes ~15–30 seconds (vs ~3–5 on 3090)
- This is normal — the model is large and we're accumulating 8 gradient steps
- At 30 hrs/week you have plenty of budget to finish in one week